<a href="https://colab.research.google.com/github/Gauravd1710/legal-doc-analyzer/blob/main/notebooks/03_iob2_conversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
import pandas as pd

BASE      = '/content/drive/MyDrive/LegalDocAnalyzer'
REPO_PATH = '/content/legal-doc-analyzer'

sys.path.append(f'{BASE}/src')

print("✅ Drive mounted")

Mounted at /content/drive
✅ Drive mounted


Loading Saved Data from Step 2 colab

In [5]:
# load CUAD mapped rows
df_useful = pd.read_json(f'{BASE}/data/raw/cuad_mapped.json')

# load entity map
with open(f'{BASE}/src/entity_map.json') as f:
    ENTITY_MAP = json.load(f)

# load splits
train_df = pd.read_json(f'{BASE}/data/processed/cuad_train.json')
val_df   = pd.read_json(f'{BASE}/data/processed/cuad_val.json')
test_df  = pd.read_json(f'{BASE}/data/processed/cuad_test.json')

print("✅ Session restored from Drive")
print(f"   df_useful : {len(df_useful)} rows")
print(f"   train_df  : {len(train_df)} rows")
print(f"   val_df    : {len(val_df)} rows")
print(f"   test_df   : {len(test_df)} rows")
print(f"\nEntity labels: {list(ENTITY_MAP.keys())}")

✅ Session restored from Drive
   df_useful : 5686 rows
   train_df  : 3980 rows
   val_df    : 853 rows
   test_df   : 853 rows

Entity labels: ['PARTY', 'DATE', 'AMOUNT', 'JURISDICTION', 'TERM']


In [4]:
!pip install transformers datasets seqeval -q

# transformers → AutoTokenizer for Legal-BERT
#                word_ids() to align labels to subwords
# datasets     → convert processed data to HuggingFace
#                Dataset format for training
# seqeval      → evaluate NER using entity-level F1

from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict, Features
from datasets import Sequence, ClassLabel, Value
import numpy as np
import re

print("✅ Libraries ready")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
✅ Libraries ready


In [ ]:
# Before writing any code, understand what IOB2 means.
# Every token in a sentence gets ONE label:
#
#   B-LABEL → Beginning of an entity
#   I-LABEL → Inside (continuation) of an entity
#   O       → Outside — not part of any entity
#
# Example sentence:
#   "Apple Inc signed a deal on January 10 2024 worth $2M"
#
#   Token       Label
#   ─────────────────
#   Apple     → B-PARTY
#   Inc       → I-PARTY
#   signed    → O
#   a         → O
#   deal      → O
#   on        → O
#   January   → B-DATE
#   10        → I-DATE
#   2024      → I-DATE
#   worth     → O
#   $2M       → B-AMOUNt


In [6]:

LABEL_LIST = [
    "O",
    "B-PARTY",    "I-PARTY",
    "B-DATE",     "I-DATE",
    "B-AMOUNT",   "I-AMOUNT",
    "B-TERM",     "I-TERM",
    "B-JURISDICTION", "I-JURISDICTION"
]

LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}

print("=== LABEL LIST ===")
for i, label in enumerate(LABEL_LIST):
    print(f"  {i:2d} → {label}")

=== LABEL LIST ===
   0 → O
   1 → B-PARTY
   2 → I-PARTY
   3 → B-DATE
   4 → I-DATE
   5 → B-AMOUNT
   6 → I-AMOUNT
   7 → B-TERM
   8 → I-TERM
   9 → B-JURISDICTION
  10 → I-JURISDICTION


In [7]:
def convert_to_iob2(context, answer_text, answer_start, entity_label):
    words = []
    word_starts = []
    word_ends   = []

    for match in re.finditer(r'\S+', context):
        words.append(match.group())
        word_starts.append(match.start())
        word_ends.append(match.end())


    answer_end = answer_start + len(answer_text)

    #  assign IOB2 label to each word
    ner_tags = []
    entity_started = False

    for i, (word, ws, we) in enumerate(zip(words, word_starts, word_ends)):
        # check if this word overlaps with the answer span
        overlaps = (ws < answer_end) and (we > answer_start)

        if overlaps:
            if not entity_started:
                # first word of entity → B-LABEL
                ner_tags.append(LABEL2ID[f"B-{entity_label}"])
                entity_started = True
            else:
                # continuation of entity → I-LABEL
                ner_tags.append(LABEL2ID[f"I-{entity_label}"])
        else:
            # not part of entity → O
            ner_tags.append(LABEL2ID["O"])
            entity_started = False

    return words, ner_tags


# ── quick test ──
test_context = "Apple Inc signed a deal on January 10 2024 worth $2M"
test_answer  = "Apple Inc"
test_start   = 0

tokens, tags = convert_to_iob2(
    test_context, test_answer, test_start, "PARTY"
)

print("=== IOB2 CONVERSION TEST ===\n")
print(f"{'Token':15s} {'Tag ID':8s} {'Tag Label'}")
print("-" * 35)
for token, tag in zip(tokens, tags):
    print(f"{token:15s} {tag:8d} {ID2LABEL[tag]}")

=== IOB2 CONVERSION TEST ===

Token           Tag ID   Tag Label
-----------------------------------
Apple                  1 B-PARTY
Inc                    2 I-PARTY
signed                 0 O
a                      0 O
deal                   0 O
on                     0 O
January                0 O
10                     0 O
2024                   0 O
worth                  0 O
$2M                    0 O


###fixed iob2

In [12]:
def convert_to_iob2(context, answer_text, answer_start, entity_label,
                    window=200):
    """
    Instead of using the full contract context (too long),
    extract a window of ~200 words around the answer span.
    This keeps sequences under 512 tokens.
    """

    answer_end = answer_start + len(answer_text)

    # ── Step 1: find word boundaries in full context ──
    import re
    words       = []
    word_starts = []
    word_ends   = []

    for match in re.finditer(r'\S+', context):
        words.append(match.group())
        word_starts.append(match.start())
        word_ends.append(match.end())

    if not words:
        return None, None

    # ── Step 2: find which words overlap with answer span ──
    answer_word_indices = []
    for i, (ws, we) in enumerate(zip(word_starts, word_ends)):
        if (ws < answer_end) and (we > answer_start):
            answer_word_indices.append(i)

    if not answer_word_indices:
        return None, None

    # ── Step 3: build a window of words around the answer ──
    first_answer_word = answer_word_indices[0]
    last_answer_word  = answer_word_indices[-1]

    # center the window around the answer
    half_window  = window // 2
    window_start = max(0, first_answer_word - half_window)
    window_end   = min(len(words), last_answer_word + half_window)

    # make sure window doesn't exceed 512 words
    if (window_end - window_start) > 512:
        window_end = window_start + 512

    window_words       = words[window_start:window_end]
    window_word_starts = word_starts[window_start:window_end]
    window_word_ends   = word_ends[window_start:window_end]

    # ── Step 4: assign IOB2 labels within the window ──
    ner_tags = []
    entity_started = False

    for ws, we in zip(window_word_starts, window_word_ends):
        overlaps = (ws < answer_end) and (we > answer_start)

        if overlaps:
            if not entity_started:
                ner_tags.append(LABEL2ID[f"B-{entity_label}"])
                entity_started = True
            else:
                ner_tags.append(LABEL2ID[f"I-{entity_label}"])
        else:
            ner_tags.append(LABEL2ID["O"])
            entity_started = False

    return window_words, ner_tags


# ── quick test ──
test_context = "Apple Inc signed a deal on January 10 2024 worth $2M"
tokens, tags = convert_to_iob2(test_context, "Apple Inc", 0, "PARTY")

print("=== WINDOWED IOB2 TEST ===\n")
print(f"{'Token':15s} {'Label'}")
print("-" * 30)
for token, tag in zip(tokens, tags):
    print(f"{token:15s} {ID2LABEL[tag]}")

=== WINDOWED IOB2 TEST ===

Token           Label
------------------------------
Apple           B-PARTY
Inc             I-PARTY
signed          O
a               O
deal            O
on              O
January         O
10              O
2024            O
worth           O
$2M             O


##**Convert entire dataset to IOB2**

In [14]:
def process_dataframe(df, split_name=""):
    all_tokens   = []
    all_ner_tags = []
    skipped      = 0

    for _, row in df.iterrows():
        try:
            # handle multiple answers per row
            answers      = row['answers']['text']
            starts       = row['answers']['answer_start']
            entity_label = row['entity_label']
            context      = row['context']

            for answer_text, answer_start in zip(answers, starts):
                if not answer_text.strip():
                    skipped += 1
                    continue

                tokens, ner_tags = convert_to_iob2(
                    context, answer_text,
                    answer_start, entity_label
                )

                if tokens is None:
                    skipped += 1
                    continue

                # now 512 word limit is handled inside
                # the window function — just check min length
                if len(tokens) < 5:
                    skipped += 1
                    continue

                all_tokens.append(tokens)
                all_ner_tags.append(ner_tags)

        except Exception as e:
            skipped += 1
            continue

    print(f"  [{split_name}] Converted: {len(all_tokens)}  |  Skipped: {skipped}")
    return all_tokens, all_ner_tags


print("Converting train set...")
train_tokens, train_tags = process_dataframe(train_df, "train")

print("Converting val set...")
val_tokens, val_tags = process_dataframe(val_df, "val")

print("Converting test set...")
test_tokens, test_tags = process_dataframe(test_df, "test")


Converting train set...
  [train] Converted: 7983  |  Skipped: 0
Converting val set...
  [val] Converted: 1685  |  Skipped: 0
Converting test set...
  [test] Converted: 1721  |  Skipped: 0


##Oversampling to solve issue of imbalance

In [20]:
from collections import Counter

def get_label_counts(tokens_list, tags_list):
    counts = Counter()
    for tags in tags_list:
        for tag in tags:
            label = ID2LABEL[tag]
            if label.startswith("B-"):
                counts[label[2:]] += 1
    return counts

print("=== BEFORE OVERSAMPLING ===")
before = get_label_counts(train_tokens, train_tags)
for label, count in sorted(before.items()):
    bar = "█" * int(count / max(before.values()) * 30)
    print(f"  {label:15s}: {count:5d}  {bar}")

# ── oversample minority classes ──
TARGET = max(before.values())

oversampled_tokens = list(train_tokens)
oversampled_tags   = list(train_tags)

for tokens, tags in zip(train_tokens, train_tags):
    entity = None
    for tag in tags:
        label = ID2LABEL[tag]
        if label.startswith("B-"):
            entity = label[2:]
            break

    if entity and before[entity] < TARGET:
        times = int(TARGET / before[entity]) - 1
        for _ in range(times):
            oversampled_tokens.append(tokens)
            oversampled_tags.append(tags)

print("\n=== AFTER OVERSAMPLING ===")
after = get_label_counts(oversampled_tokens, oversampled_tags)
for label, count in sorted(after.items()):
    bar = "█" * int(count / max(after.values()) * 30)
    print(f"  {label:15s}: {count:5d}  {bar}")

print(f"\nTotal train examples: {len(oversampled_tokens)}")

=== BEFORE OVERSAMPLING ===
  AMOUNT         :  1848  ████████████████████
  DATE           :  1234  █████████████
  JURISDICTION   :   322  ███
  PARTY          :  2739  ██████████████████████████████
  TERM           :  1840  ████████████████████

=== AFTER OVERSAMPLING ===
  AMOUNT         :  1848  ████████████████████
  DATE           :  2468  ███████████████████████████
  JURISDICTION   :  2576  ████████████████████████████
  PARTY          :  2739  ██████████████████████████████
  TERM           :  1840  ████████████████████

Total train examples: 11471


In [21]:
import torch, json

# class weights to compensate for JURISDICTION being oversampled
# higher weight = model pays more attention to that class during training

class_weights = []
after = get_label_counts(oversampled_tokens, oversampled_tags)
total = sum(after.values())

for label in LABEL_LIST:
    if label == "O":
        class_weights.append(0.1)   # O is everywhere — downweight it
    elif label.startswith("B-") or label.startswith("I-"):
        entity = label[2:]
        count  = after.get(entity, 1)
        weight = round(total / (len(LABEL_LIST) * count), 3)
        class_weights.append(weight)
    else:
        class_weights.append(1.0)

print("=== CLASS WEIGHTS ===")
for label, w in zip(LABEL_LIST, class_weights):
    print(f"  {label:22s}: {w:.3f}")

with open(f'{BASE}/src/class_weights.json', 'w') as f:
    json.dump(class_weights, f)

print("\n✅ Saved to src/class_weights.json")
print("   These will be used in Step 4 training loop")

=== CLASS WEIGHTS ===
  O                     : 0.100
  B-PARTY               : 0.381
  I-PARTY               : 0.381
  B-DATE                : 0.423
  I-DATE                : 0.423
  B-AMOUNT              : 0.564
  I-AMOUNT              : 0.564
  B-TERM                : 0.567
  I-TERM                : 0.567
  B-JURISDICTION        : 0.405
  I-JURISDICTION        : 0.405

✅ Saved to src/class_weights.json
   These will be used in Step 4 training loop


##Build HuggingFace Dataset

In [22]:
from datasets import Dataset, DatasetDict

def make_hf_dataset(tokens_list, tags_list):
    return Dataset.from_dict({
        "tokens"  : tokens_list,
        "ner_tags": tags_list
    })

dataset = DatasetDict({
    "train"     : make_hf_dataset(oversampled_tokens, oversampled_tags),
    "validation": make_hf_dataset(val_tokens,         val_tags),
    "test"      : make_hf_dataset(test_tokens,         test_tags)
})

print("=== HUGGINGFACE DATASET ===")
print(dataset)

print(f"\nSample from train:")
sample = dataset['train'][0]
print(f"  tokens  : {sample['tokens'][:8]}")
print(f"  ner_tags: {sample['ner_tags'][:8]}")

=== HUGGINGFACE DATASET ===
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 11471
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1685
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1721
    })
})

Sample from train:
  tokens  : ['has', 'been', 'requested', 'for', 'redacted', 'portions', 'of', 'this']
  ner_tags: [0, 0, 0, 0, 0, 0, 0, 0]


##Tokenizing with legal bert and analyzing

In [23]:
from transformers import AutoTokenizer

print("Loading Legal-BERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    "nlpaueb/legal-bert-base-uncased"
)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=512,
        is_split_into_words=True
    )

    aligned_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids         = tokenized.word_ids(batch_index=i)
        label_ids        = []
        previous_word_id = None

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)          # [CLS] [SEP]
            elif word_id != previous_word_id:
                label_ids.append(labels[word_id]) # first subword
            else:
                label_ids.append(-100)            # continuation subword

            previous_word_id = word_id

        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels
    return tokenized

print("Tokenizing dataset... (3-4 mins)")

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=["tokens", "ner_tags"]
)

print("\n=== TOKENIZED DATASET ===")
print(tokenized_dataset)

# sanity check lengths match
s = tokenized_dataset['train'][0]
print(f"\ninput_ids length : {len(s['input_ids'])}")
print(f"labels length    : {len(s['labels'])}")
print(f"lengths match    : {len(s['input_ids']) == len(s['labels'])}")

Loading Legal-BERT tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Tokenizing dataset... (3-4 mins)


Map:   0%|          | 0/11471 [00:00<?, ? examples/s]

Map:   0%|          | 0/1685 [00:00<?, ? examples/s]

Map:   0%|          | 0/1721 [00:00<?, ? examples/s]


=== TOKENIZED DATASET ===
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 11471
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1685
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1721
    })
})

input_ids length : 300
labels length    : 300
lengths match    : True


##Verifying sample

In [24]:
sample    = tokenized_dataset['train'][5]
input_ids = sample['input_ids']
labels    = sample['labels']
tokens    = tokenizer.convert_ids_to_tokens(input_ids)

print("=== END TO END VERIFICATION ===\n")
print(f"{'Subword Token':22s} {'Label'}")
print("-" * 40)

for token, label_id in zip(tokens[:35], labels[:35]):
    if label_id == -100:
        label_str = "[-100]"
    else:
        label_str = ID2LABEL[label_id]
    marker = " ←" if label_id not in [-100, 0] else ""
    print(f"{token:22s} {label_str}{marker}")

print("\n✅ Correct if:")
print("  B-/I- labels appear on entity subwords  ←")
print("  [-100] on ## subwords and [CLS]/[SEP]")
print("  O on all non-entity words")

=== END TO END VERIFICATION ===

Subword Token          Label
----------------------------------------
[CLS]                  [-100]
franchise              O
##d                    [-100]
business               O
'                      [-100]
operation              O
and                    O
maintaining            O
the                    O
good                   O
##will                 [-100]
of                     O
the                    O
business               O
.                      [-100]
16                     O
.                      [-100]
4                      [-100]
.                      [-100]
liquidated             O
damages                O
.                      [-100]
if                     O
this                   O
agreement              O
is                     O
terminated             O
due                    O
to                     O
your                   O
default                O
,                      [-100]
you                    O
must                  

In [26]:
# find first sample that contains a B- label
entity_sample_idx = None
for idx in range(len(tokenized_dataset['train'])):
    labels = tokenized_dataset['train'][idx]['labels']
    if any(l not in [-100, 0] for l in labels):
        entity_sample_idx = idx
        break

print(f"First sample with entity labels: index {entity_sample_idx}\n")

sample    = tokenized_dataset['train'][entity_sample_idx]
input_ids = sample['input_ids']
labels    = sample['labels']
tokens    = tokenizer.convert_ids_to_tokens(input_ids)

print(f"{'Subword Token':25s} {'Label'}")
print("-" * 45)

for token, label_id in zip(tokens[:40], labels[:40]):
    if label_id == -100:
        label_str = "[-100]"
    elif label_id == 0:
        label_str = "O"
    else:
        label_str = ID2LABEL[label_id]
    marker = " ←" if label_id not in [-100, 0] else ""
    print(f"{token:25s} {label_str}{marker}")

# count entity labels in this sample
entity_labels = [ID2LABEL[l] for l in labels if l not in [-100, 0]]
print(f"\nEntity labels found: {entity_labels}")


First sample with entity labels: index 0

Subword Token             Label
---------------------------------------------
[CLS]                     [-100]
has                       O
been                      O
requested                 O
for                       O
redacte                   O
##d                       [-100]
portion                   O
##s                       [-100]
of                        O
this                      O
exhibit                   O
.                         [-100]
this                      O
copy                      O
omit                      O
##s                       [-100]
the                       O
information               O
subject                   O
to                        O
the                       O
confidentiality           O
request                   O
.                         [-100]
omissions                 O
are                       O
designated                O
as                        O
[                         O
*         

<h2 style="color:red">Full sanity check for subword label alignment</h2>

In [28]:
from collections import Counter

# Use the correct variable
issues = 0
subword_misalignment = 0
entity_counts = Counter()

for split in ["train", "validation", "test"]:
    dataset = tokenized_dataset[split]  # <-- singular name
    for i, example in enumerate(dataset):
        input_len = len(example['input_ids'])
        label_len = len(example['labels'])
        if input_len != label_len:
            print(f"Length mismatch in {split} sample {i}: input={input_len}, labels={label_len}")
            issues += 1

        # Convert input_ids to tokens
        tokens = tokenizer.convert_ids_to_tokens(example['input_ids'])
        for t, l in zip(tokens, example['labels']):
            if l != -100 and t.startswith("##"):
                print(f"Subword with label != -100 in {split} sample {i}: token={t}, label={l}")
                subword_misalignment += 1

            # Count all entity labels
            if l != -100 and l != 0:
                entity_counts[l] += 1

print("Total length mismatches:", issues)
print("Total subword misalignments:", subword_misalignment)
print("Entity label counts:", {ID2LABEL[k]: v for k, v in entity_counts.items()})

Total length mismatches: 0
Total subword misalignments: 0
Entity label counts: {'B-AMOUNT': 2480, 'I-AMOUNT': 143997, 'B-JURISDICTION': 2718, 'I-JURISDICTION': 89246, 'B-DATE': 2948, 'I-DATE': 66827, 'B-TERM': 2679, 'I-TERM': 141625, 'B-PARTY': 3990, 'I-PARTY': 72331}


###Saving to drive

In [31]:
import json

# save tokenized dataset
save_path = f'{BASE}/data/processed/tokenized_dataset'
tokenized_dataset.save_to_disk(save_path)
print(f"✅ Tokenized dataset saved")

# save label config
label_config = {
    "label_list": LABEL_LIST,
    "label2id"  : LABEL2ID,
    "id2label"  : {str(k): v for k, v in ID2LABEL.items()},
    "num_labels": len(LABEL_LIST)
}
with open(f'{BASE}/src/label_config.json', 'w') as f:
    json.dump(label_config, f, indent=2)
print(f"✅ Label config saved")

# save raw iob2 before tokenization
iob2_data = {
    "train"     : {"tokens": oversampled_tokens, "ner_tags": oversampled_tags},
    "validation": {"tokens": val_tokens,         "ner_tags": val_tags},
    "test"      : {"tokens": test_tokens,        "ner_tags": test_tags}
}
with open(f'{BASE}/data/processed/iob2_data.json', 'w') as f:
    json.dump(iob2_data, f)
print(f"✅ Raw IOB2 data saved")

print(f"\n=== FINAL DATASET SIZES ===")
print(f"  Train      : {len(tokenized_dataset['train'])}")
print(f"  Validation : {len(tokenized_dataset['validation'])}")
print(f"  Test       : {len(tokenized_dataset['test'])}  ← FROZEN")

print(f"\n=== ALL FILES SAVED TO DRIVE ===")
print(f"  data/processed/tokenized_dataset/")
print(f"  data/processed/iob2_data.json")
print(f"  src/label_config.json")
print(f"  src/class_weights.json")

Saving the dataset (0/1 shards):   0%|          | 0/11471 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1685 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1721 [00:00<?, ? examples/s]

✅ Tokenized dataset saved
✅ Label config saved
✅ Raw IOB2 data saved

=== FINAL DATASET SIZES ===
  Train      : 11471
  Validation : 1685
  Test       : 1721  ← FROZEN

=== ALL FILES SAVED TO DRIVE ===
  data/processed/tokenized_dataset/
  data/processed/iob2_data.json
  src/label_config.json
  src/class_weights.json


##diagnosis

In [9]:
# check what's actually inside answers column
sample = train_df.iloc[0]

print("=== RAW ANSWER STRUCTURE ===")
print(type(sample['answers']))
print(sample['answers'])

print("\n=== ANSWER TEXT ===")
print(repr(sample['answers']['text']))

print("\n=== ANSWER START ===")
print(repr(sample['answers']['answer_start']))

print("\n=== CONTEXT SNIPPET ===")
print(repr(sample['context'][:300]))

=== RAW ANSWER STRUCTURE ===
<class 'dict'>
{'text': ['MediWound undertakes to order at least [***]% of the Annual Forecast per each year.', 'CBC shall maintain, at all times, manufacture and supply capacity of at least [***]% of the Annual Forecast and shall maintain, in coordination with MediWound, inventory of Bromelain SP at its premises of (i) at least [***]% of the applicable Annual Forecast; and (ii) all Bromelain SP components and materials ("the BSP Components and Materials") needed for the manufacture and supply of the Bromelain SP such that CBC can guarantee continuous supply of the Bromelain SP in accordance with MediWound\'s complete Annual Forecasts.', 'Purchase orders issued by MediWound to CBC for quantities within the [***]% of the Annual Forecast shall be binding upon CBC and shall be deemed accepted upon delivery of the purchase order to CBC.'], 'answer_start': [25411, 25497, 26477]}

=== ANSWER TEXT ===
['MediWound undertakes to order at least [***]% of the Annual F

In [10]:
# verify character position is correct
sample = train_df.iloc[0]

answer_text  = sample['answers']['text'][0]
answer_start = sample['answers']['answer_start'][0]
context      = sample['context']

print(f"Answer text    : {repr(answer_text)}")
print(f"Answer start   : {answer_start}")
print(f"Context length : {len(context)}")

# check what's at that position
if answer_start < len(context):
    extracted = context[answer_start : answer_start + len(answer_text)]
    print(f"\nExtracted at position : {repr(extracted)}")
    print(f"Matches answer        : {extracted == answer_text}")
else:
    print(f"\n❌ answer_start ({answer_start}) > context length ({len(context)})")
    print("This is the bug — positions don't match the context")

Answer text    : 'MediWound undertakes to order at least [***]% of the Annual Forecast per each year.'
Answer start   : 25411
Context length : 49915

Extracted at position : 'MediWound undertakes to order at least [***]% of the Annual Forecast per each year.'
Matches answer        : True


In [11]:
# trace exactly why rows are being skipped

reasons = {
    'too_short'      : 0,
    'too_long'       : 0,
    'position_mismatch': 0,
    'empty_answer'   : 0,
    'other_error'    : 0,
    'success'        : 0
}

for _, row in train_df.head(50).iterrows():
    try:
        answer_text  = row['answers']['text'][0]
        answer_start = row['answers']['answer_start'][0]
        context      = row['context']

        # check position validity
        if answer_start >= len(context):
            reasons['position_mismatch'] += 1
            continue

        extracted = context[answer_start : answer_start + len(answer_text)]
        if extracted != answer_text:
            reasons['position_mismatch'] += 1
            continue

        tokens, ner_tags = convert_to_iob2(
            context, answer_text, answer_start, row['entity_label']
        )

        if len(tokens) < 5:
            reasons['too_short'] += 1
        elif len(tokens) > 512:
            reasons['too_long'] += 1
        else:
            reasons['success'] += 1

    except Exception as e:
        reasons['other_error'] += 1
        print(f"Error: {e}")

print("=== SKIP REASONS (first 50 rows) ===")
for reason, count in reasons.items():
    print(f"  {reason:20s}: {count}")

=== SKIP REASONS (first 50 rows) ===
  too_short           : 0
  too_long            : 50
  position_mismatch   : 0
  empty_answer        : 0
  other_error         : 0
  success             : 0
